# Risk Assessment

Land Suitability Analysis (LSA) and risk assessment are two distinct but related methodologies, and the use of one rather than the other depends on the context and what we want to highlight. If we are looking for to show the potential of a given land use, LSA is more appropriate, but if we want to assess the vulnerability against a specific hazard, we will use risk assessment.

While LSAPy has been primarily designed for LSA, its features can also be used to perform risk assessments. This short example aims to illustrate how LSAPy can be used to perform a risk assessment, using as example the risk of Botrytis, a fungal disease that affects many crops, on wine grapes.

## Indicator

The development of Botrytis is favored by humid and warm conditions. However, for simplicity, we will use the total precipitation in March and April as a proxy for the risk of Botrytis, as in the [NZLUSDB project](https://baptistehamon.github.io/nzlusdb/). The indicator is available in the [lsapy-data](https://github.com/baptistehamon/lsapy-data) repository under the name `prcptot_0301_0430`, and can be loaded using the `lsapy.tutorial.load_dataset` function.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lsapy.standardize as lstd
from lsapy import SuitabilityCriteria, tutorial

data = tutorial.load_dataset("prcptot_0301-0430")
data

Let's have a look at the data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in zip([2, 3, 4], axes, strict=True):
    data["prcptot"].isel(time=i).plot(ax=ax, vmin=0, vmax=800)
plt.tight_layout()

While the rainfall patterns are relatively similar across the three years, there is spatial variability in the total precipitation, which can influence the development of Botrytis. The next step is to assess its risk.

## Risk Assessment

The risk can basically be seen as the inverse of the suitability. We can therefore use the same standardization function for suitability analysis, but by inverting it. In case of risk assessment, such standardization function can be called vulnerability function (or curve). 
In this example, we will use the function and parameters from the NZLUSDB project, as follows:

In [ ]:
x = np.linspace(0, 400, 100)  # 0 to 400 mm of precipitation
suitability = lstd.vetharaniam2022_eq5(x, 0.9311, 160)
risk = lstd.vetharaniam2022_eq5(x, -0.9311, 160)
# or
# risk = 1 - suitability

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(x, suitability, label="Suitability")
ax.plot(x, risk, label="Risk")
ax.set_title("Suitability and Risk curves for Botrytis on wine grapes")
ax.set_xlabel("Total precipitation (mm)")
ax.set_ylabel("Suitability/Risk score")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)

We can now apply the function to the data using the `SuitabilityCriteria` class, and then compute the risk.

In [ ]:
sc = SuitabilityCriteria(
    name="Botrytis",
    long_name="Risk of Botrytis",
    indicator=data["prcptot"],
    func=lstd.vetharaniam2022_eq5,
    fparams={"a": -0.9311, "b": 160},
)

risk = sc.compute()

We can visualize the spatial distribution of the risk of Botrytis on wine grapes for the three given years, and the mean risk over the five years.

In [ ]:
mean = risk.mean(dim="time")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, ax in zip([2, 3, 4], axes[:3], strict=True):
    risk.isel(time=i).plot(ax=ax, vmin=0, vmax=1)
    ax.set_title(f"{risk.time.isel(time=i).dt.year.values}")
mean.plot(ax=axes[3], vmin=0, vmax=1)
axes[3].set_title("2000-2004 Mean")
plt.tight_layout()

The risk of Botrytis on wine grapes is highly variable across the three years, with the North Island being almost entirely exposed to high risk in 2002 and 2004, while the risk is limited in 2003. The mean across the five years shows that some areas are consistently exposed to high risk, such as the some parts of the West, East, and South of the North Island.